### 문항 1 PyMySQL 모듈 기반 카페 DB(cafe_db) 연동  커넥터 클래스 구현

Python에서 MariaDB의 cafe_db 데이터베이스에 연결하고, 쿼리 조회, 수정, 예외 처리 및 자원 반납을 안전하게 수행할 수 있는 CafeDBManager 클래스를 작성하세요.

In [ ]:
# [실습 요구사항]

# pymysql 라이브러리를 설치 및 import 하세요 (pip install pymysql).

In [3]:
%pip install pymysql python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv
import pymysql
from pymysql.cursors import DictCursor

# .env 설정 파일 불러오기
load_dotenv()

True

In [2]:
# DB config는 클래스 외부에 작성 
DB_CONFIG = {
    'host': os.environ.get('DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('DB_PORT', '3306')),
    'user': os.environ.get('DB_USER', 'analyst'),
    'password': os.environ.get('DB_PASSWORD', ''),
    'database': os.environ.get('DB_NAME', 'shop_db'),
    'charset': 'utf8mb4',
    'cursorclass': DictCursor,
    'autocommit': False,
}

In [ ]:
# 클래스 생성자(__init__)에서 cafe_db 접속 정보(host, user, password, db, port)를 전달받아 DictCursor 기반 커넥션을 설정하는 구조를 구현하세요.

# SELECT 쿼리를 수행하고 딕셔너리 리스트 결과를 반환하는 execute_query(sql, params) 메서드를 작성하세요.

# INSERT/UPDATE/DELETE 쿼리를 수행하고 commit()을 자동 처리하는 execute_update(sql, params) 메서드를 작성하세요.

# try-except-finally 구조로 에러 발생 시 rollback() 및 커넥션 자원 반납(close())이 수행되도록 구현하세요.

In [28]:
class CafeDBManager:
    def __init__(self, config):
        self.config = config

    def _connect(self):
        return pymysql.connect(**self.config)

    def execute_query(self, sql, params=None):
        # SELECT 쿼리를 수행하고 딕셔너리 리스트 반환
        conn = self._connect()

        try:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                result = cur.fetchall()
                return result
        except Exception as e:
            print(f'[Query Error] {e}')
            return []
        finally:
            conn.close()

    def execute_update(self, sql, params=None):
        # INSERT/UPDATE/DELETE 쿼리를 실행.
        conn = self._connect()

        try:
            with conn.cursor() as cur:
                affected_rows = cur.execute(sql, params)
            conn.commit()
            return affected_rows
        except Exception as e:
            conn.rollback()
            print(f'[UPDATE Error & Rollback] {e}')
            return 0
        finally:
            conn.close()

if __name__ == '__main__':
    db = CafeDBManager(DB_CONFIG)

    sql= """
        SELECT store_id, store_nm, opened_dt FROM tb_store
    """
    stores = db.execute_query(sql)

    print('===체인점 매장 목록===')

    for store in stores:
        print(
        store['store_id'],
        store['store_nm'],
        f"개점일: {store['opened_dt']:%Y-%m-%d}, "
        )

===체인점 매장 목록===
1 강남역점 개점일: 2019-03-15, 
2 홍대입구점 개점일: 2020-07-01, 
3 판교테크노점 개점일: 2021-05-20, 
4 해운대점 개점일: 2020-11-10, 
5 대전둔산점 개점일: 2022-02-14, 
6 광주충장로점 개점일: 2022-09-05, 
7 수원영통점 개점일: 2023-04-18, 
8 인천송도점 개점일: 2023-08-22, 


### 분석 1 매장별 매출 및 객단가(AOV) 진단

In [30]:
def db_query(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()

conn = pymysql.connect(**DB_CONFIG)

In [12]:
# SQL 요구사항: 매장별 매장명, 독립 주문건수, 총 매출액을 조회하세요. (1차 집계 목표: 8행)

import pandas as pd

sql_1 = """
    SELECT 
        s.store_nm,
        COUNT(DISTINCT o.order_id) AS cnt,
        SUM(oi.qty*oi.unit_price) AS total
    FROM tb_order o 
        JOIN tb_order_item oi ON oi.order_id=o.order_id
        JOIN tb_store s ON s.store_id= o.store_id
    GROUP BY s.store_nm;

"""
# Python 요구사항:
# 수집된 데이터프레임에서 총 매출액을 독립 주문건수로 나누어 평균 객단가(AOV) 컬럼을 계산하세요.
# 총 매출액 기준으로 내림차순 정렬한 후, 매출 1위 매장의 성과 및 분석 의견을 프린트 또는 주석으로 작성하세요.

df1 = pd.DataFrame(db_query(conn,sql_1))
df1['객단가(AOV)']= (df1['total']/df1['cnt']).round(0)
df1 = df1.sort_values(by='total', ascending=False)

In [13]:
df1

,store_nm,cnt,total,객단가(AOV)
2,대전둔산점,15000,192885200.00,12859
3,수원영통점,15000,192884700.00,12859
0,강남역점,15000,192883000.00,12859
5,판교테크노점,15000,192882600.00,12859
7,홍대입구점,15000,170938000.00,11396
1,광주충장로점,15000,170935800.00,11396
4,인천송도점,15000,170934400.00,11396
6,해운대점,15000,170929800.00,11395


In [14]:
df1.info()

<class 'pandas.DataFrame'>
Index: 8 entries, 2 to 6
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   store_nm  8 non-null      str   
 1   cnt       8 non-null      int64 
 2   total     8 non-null      object
 3   객단가(AOV)  8 non-null      object
dtypes: int64(1), object(2), str(1)
memory usage: 320.0+ bytes


In [15]:
top_store = df1.iloc[0]

In [22]:
print(f'{top_store['store_nm']} 매장이 매출 {int(top_store['total']):,}원 을 올려 전체 매장 1위를 달성하였습니다.')

대전둔산점 매장이 매출 192,885,200원 을 올려 전체 매장 1위를 달성하였습니다.


### 분석 2 시간대별 주문 피크 타임 분석

In [23]:
# SQL 요구사항: 영업 시간대(0~23시)별 주문 시간대, 총 주문건수를 조회하세요. (1차 집계 목표: 13행)

sql_2 = """
    SELECT HOUR(order_dt) AS hour,
        count(*) AS hour_order_total
    FROM tb_order
    GROUP BY HOUR(order_dt)
"""

# Python 요구사항:
# 수집된 데이터프레임에서 주문건수가 가장 많은 상위 3개 피크 시간대를 추출하세요.
# 가장 주문이 몰리는 피크 시간대를 바탕으로 매장 파트타임 인력 배치에 대한 제언을 주석으로 작성하세요.
df2 = pd.DataFrame(db_query(conn,sql_2))
peak_t = df2.sort_values(by='hour_order_total', ascending=False).head(3)

In [24]:
peak_t

,hour,hour_order_total
2,9,18000
3,10,12000
1,8,12000


In [28]:
print('주문 건수 Top3 시간대')

for idx, row in peak_t.iterrows():
    print(f'{row['hour']}시에 총 {int(row['hour_order_total']):,} 주문 발생')

주문 건수 Top3 시간대
9시에 총 18,000 주문 발생
10시에 총 12,000 주문 발생
8시에 총 12,000 주문 발생


### 분석 3 메뉴 카테고리별 매출 점유율 분석

In [44]:
# SQL 요구사항: 메뉴 카테고리별 카테고리명, 총 매출액, 총 판매수량을 조회하세요. (1차 집계 목표: 5행)

sql_3 = """ 
    SELECT mc.category_nm,
        IFNULL(SUM(oi.qty*oi.unit_price),0) AS total_category,
        IFNULL(SUM(oi.qty), 0) AS total_cnt
    FROM tb_menu_category mc
        JOIN tb_menu m ON mc.category_id=m.category_id
        LEFT JOIN tb_order_item oi ON m.menu_id=oi.menu_id
    GROUP BY mc.category_nm
"""

# Python 요구사항:
# 수집된 데이터프레임에서 각 카테고리별 매출액을 전체 총매출액으로 나눈 뒤 100을 곱하여 매출 점유율(%)을 구하세요.
# 매출 점유율 기준으로 내림차순 정렬한 뒤, 가장 높게 나타난 주력 카테고리에 대한 영업 인사이트를 프린트 또는 주석으로 작성하세요.
df3 = pd.DataFrame(db_query(conn,sql_3))
df3['매출점유율(%)']= ((df3['total_category'] / df3['total_category'].sum())*100).round(1)
df3=df3.sort_values(by='매출점유율(%)', ascending=False)
top_category = df3.iloc[0]
print(f'{top_category['category_nm']}가 매장 전체 매출에서 {top_category['매출점유율(%)']}%를 차지하는 주력 카테고리라 할 수 있다.')

커피가 매장 전체 매출에서 34.5%를 차지하는 주력 카테고리라 할 수 있다.
